# Load test Pha 2 — Bank Account (protocol v2)

Đo theo `docs/phase2/giao-thuc-do.md`: đăng nhập, xem tài khoản, rút tiền một tài khoản nóng, rút tiền rải trên 20 tài khoản, chuyển khoản và tải hỗn hợp đăng nhập cùng xem tài khoản.
API, PostgreSQL 15 và k6 cùng chạy trên Kaggle CPU. Chọn **Accelerator: None**, **Internet: On**.

Kịch bản và thông số đọc từ repo đã checkout (`test/performance/kaggle-load.js`, `test/performance/protocol-v2.json`), notebook không giữ bản sao.

Chạy thử với `MODE = 'smoke'` trước. Sau đó đặt `MODE = 'full'` trong ô mới và chạy lại phần 9–11.

## 1. Khai báo
`REPO_URL` và `COMMIT` phải trỏ đúng fork và SHA cần đo. `WORKLOADS = None` chạy mọi kịch bản; đặt danh sách, ví dụ `['debit', 'login', 'mixed']`, để chạy trước vài ô.

In [ ]:
import os
import json
import time
import uuid
import shutil
import secrets
import subprocess
from pathlib import Path

MODE = 'smoke'
REPO_URL = 'https://github.com/datdooo/SA_BankAccount.git'
COMMIT = 'ĐIỀN_SHA_CẦN_ĐO'
PROTOCOL = 'test/performance/protocol-v2.json'
WORKLOADS = None

ROOT = Path('/tmp/bank-source') / COMMIT
OUT = Path('/kaggle/working') / ('bank-test-' + time.strftime('%Y%m%d-%H%M%S'))
OUT.mkdir(parents=True, exist_ok=True)
assert len(COMMIT) == 40, 'Set COMMIT to the full SHA being measured'
BASE = 'http://127.0.0.1:3000/api/v1'
DB_URL = 'postgresql://postgres@127.0.0.1:55432/bank_load_test'

def run(command, cwd=None):
    return subprocess.run(command, cwd=cwd, check=True, text=True)

def read(command):
    return subprocess.check_output(command, text=True).strip()

Thông số đo nằm trong `PROTOCOL` của repo, được đọc ở phần 3 sau khi checkout. Mỗi client nghỉ 0,1 giây sau một request.

In [ ]:
# Settings are loaded from the checked-out repo in section 3.

## 2. Cài công cụ
Dùng Node 24.22.0, PostgreSQL 15 và k6 1.3.0. PostgreSQL được cài từ kho PGDG; phiên bản thực tế được lưu trong environment.json. Lần đo ngày 28/10/2026 (UTC) dùng PostgreSQL 15.19. psutil dùng để đọc CPU/RAM của API theo PID.

In [ ]:
%%bash
set -e
apt-get update -qq
apt-get install -y -qq git curl xz-utils ca-certificates openssl

. /etc/os-release
mkdir -p /usr/share/postgresql-common/pgdg
curl -fL --retry 3 https://www.postgresql.org/media/keys/ACCC4CF8.asc -o /usr/share/postgresql-common/pgdg/apt.postgresql.org.asc
echo "deb [signed-by=/usr/share/postgresql-common/pgdg/apt.postgresql.org.asc] https://apt.postgresql.org/pub/repos/apt ${VERSION_CODENAME}-pgdg main" > /etc/apt/sources.list.d/pgdg.list
apt-get update -qq
apt-get install -y -qq postgresql-15 postgresql-client-15
python -m pip install -q psutil==7.2.2

mkdir -p /tmp/bank-tools
cd /tmp/bank-tools
curl -fL --retry 3 https://nodejs.org/dist/v24.22.0/node-v24.22.0-linux-x64.tar.xz -o node.tar.xz
tar -xf node.tar.xz
curl -fL --retry 3 https://github.com/grafana/k6/releases/download/v1.3.0/k6-v1.3.0-linux-amd64.tar.gz -o k6.tar.gz
tar -xzf k6.tar.gz


## 3. Lấy source và build
Phải cài cả devDependencies vì lệnh build cần Nest CLI.

In [ ]:
import requests
import psutil
import pandas as pd

os.environ['PATH'] = '/tmp/bank-tools/node-v24.22.0-linux-x64/bin:/tmp/bank-tools/k6-v1.3.0-linux-amd64:' + os.environ['PATH']
os.environ.update({
    'DATABASE_URL': DB_URL, 'NODE_ENV': 'production', 'PORT': '3000',
    'JWT_ACCESS_SECRET': secrets.token_hex(32), 'JWT_ACCESS_TTL': '2h',
    'BCRYPT_ROUNDS': '12', 'UV_THREADPOOL_SIZE': '4',
})
if not ROOT.exists():
    run(['git', 'clone', REPO_URL, str(ROOT)])
run(['git', 'checkout', '--detach', COMMIT], ROOT)
assert read(['git', '-C', str(ROOT), 'rev-parse', 'HEAD']) == COMMIT
run(['npm', 'ci', '--include=dev'], ROOT)
run(['npx', '--no-install', 'prisma', 'generate'], ROOT)
run(['npm', 'run', 'build'], ROOT)
shutil.copy(ROOT / 'package-lock.json', OUT)
settings = json.loads((ROOT / PROTOCOL).read_text())
Path('/tmp/bank-protocol.json').write_text(json.dumps(settings))
(OUT / 'protocol.json').write_text(json.dumps(settings, indent=2))
print('Protocol:', settings['id'])
run(['git', 'archive', '--format=tar', '--output=' + str(OUT / 'source.tar'), COMMIT], ROOT)

## 4. Tạo database test
Database `bank_load_test` chỉ dành cho bài này. Các lượt đo sẽ xóa dữ liệu test rồi tạo lại.

In [ ]:
PG_BIN = '/usr/lib/postgresql/15/bin/'
PG_DATA = '/tmp/bank-postgres-15'
Path(PG_DATA).mkdir(exist_ok=True)
run(['chown', 'postgres:postgres', PG_DATA])
if not Path(PG_DATA, 'PG_VERSION').exists():
    run(['runuser', '-u', 'postgres', '--', PG_BIN + 'initdb', '-D', PG_DATA,
         '-A', 'trust', '--encoding=UTF8', '--locale=C'])
status = subprocess.run(['runuser', '-u', 'postgres', '--', PG_BIN + 'pg_ctl', '-D', PG_DATA, 'status'], capture_output=True)
if status.returncode != 0:
    run(['runuser', '-u', 'postgres', '--', PG_BIN + 'pg_ctl', '-D', PG_DATA,
         '-l', PG_DATA + '/server.log', '-o', '-h 127.0.0.1 -p 55432', '-w', 'start'])

def sql(query, database=DB_URL):
    return read([PG_BIN + 'psql', database, '-v', 'ON_ERROR_STOP=1', '-Atc', query])

admin = 'postgresql://postgres@127.0.0.1:55432/postgres'
assert 150000 <= int(sql('SHOW server_version_num', admin)) < 160000, 'Expected PostgreSQL 15; start a new Kaggle session'
if sql("SELECT 1 FROM pg_database WHERE datname='bank_load_test'", admin) != '1':
    sql('CREATE DATABASE bank_load_test', admin)
run(['npx', '--no-install', 'prisma', 'migrate', 'deploy'], ROOT)
print('Database ready:', sql('SHOW server_version'))


## 5. Chạy API
Request chưa có token trả 401 là đúng. Nếu app không chạy được thì xem `app.log`.

In [ ]:
if 'app' in globals() and app.poll() is None:
    app.terminate()
    app.wait(timeout=15)
app_log = open(OUT / 'app.log', 'w')
app = subprocess.Popen(['node', 'dist/main.js'], cwd=ROOT,
                       stdout=app_log, stderr=subprocess.STDOUT)
ready = False
for _ in range(60):
    if app.poll() is not None:
        break
    try:
        ready = requests.get(BASE + '/accounts', timeout=1).status_code == 401
        if ready:
            break
    except requests.RequestException:
        pass
    time.sleep(1)
assert ready, (OUT / 'app.log').read_text()[-4000:]
print('API ready, PID:', app.pid)


## 6. Ghi lại môi trường
Lưu cấu hình thực tế để biết số liệu được đo trên máy và phiên bản nào.

In [ ]:
environment = {
    'commit': COMMIT, 'repo': REPO_URL, 'protocol': settings['id'],
    'cpu': read(['lscpu']), 'memory': read(['free', '-b']),
    'node': read(['node', '--version']), 'k6': read(['k6', 'version']),
    'postgres': sql('SELECT version()'), 'psutil': psutil.__version__,
    'bcrypt_rounds': 12, 'uv_threadpool_size': 4,
    'topology': 'API, PostgreSQL and k6 on the same Kaggle CPU session',
    'postgres_settings': sql("SELECT name || '=' || setting FROM pg_settings WHERE name IN ('max_connections','shared_buffers','work_mem','fsync','synchronous_commit') ORDER BY name"),
}
for name in ['cpu.max', 'cpuset.cpus.effective', 'memory.max']:
    file = Path('/sys/fs/cgroup') / name
    environment[name] = file.read_text().strip() if file.exists() else 'unavailable'
(OUT / 'environment.json').write_text(json.dumps(environment, indent=2))
print(json.dumps(environment, indent=2))

## 7. Script k6
Script nằm trong repo ở `test/performance/kaggle-load.js`. Notebook chạy đúng file của commit đang đo và lưu một bản vào thư mục kết quả.

In [ ]:
K6_SCRIPT = ROOT / 'test/performance/kaggle-load.js'
print(K6_SCRIPT.read_text())

In [ ]:
shutil.copy(K6_SCRIPT, OUT / 'kaggle-load.js')

## 8. Chuẩn bị dữ liệu và hàm chạy test
Mỗi lượt có 1 user, 20 account; mỗi account ban đầu có 1 tỷ VND.

In [ ]:
PASSWORD = 'KaggleLoad123!'
INITIAL = int(settings['fixture']['initial_balance_per_account'])

def api(method, path, token=None, body=None, status=200):
    headers = {'Authorization': 'Bearer ' + token} if token else {}
    response = requests.request(method, BASE + path, headers=headers, json=body, timeout=35)
    assert response.status_code == status, response.text
    data = response.json()
    assert data['success'] is True, data
    return data['data']

def create_data():
    assert DB_URL == 'postgresql://postgres@127.0.0.1:55432/bank_load_test'
    sql('TRUNCATE financial_transactions, transfers, bank_accounts, users')
    email = uuid.uuid4().hex + '@example.test'
    api('POST', '/auth/register', body={'fullName': 'Load Test', 'email': email,
        'password': PASSWORD, 'confirmPassword': PASSWORD}, status=201)
    token = api('POST', '/auth/login', body={'email': email, 'password': PASSWORD})['accessToken']
    accounts = []
    for i in range(settings['fixture']['accounts']):
        account = api('POST', '/accounts', token,
                      {'accountName': 'Account ' + str(i), 'accountType': 'PAYMENT',
                       'initialDeposit': str(INITIAL)}, status=201)
        accounts.append(account)
    data = {'email': email, 'password': PASSWORD, 'token': token,
            'accountIds': [a['id'] for a in accounts], 'accountNumbers': [a['accountNumber'] for a in accounts]}
    Path('/tmp/bank-data.json').write_text(json.dumps(data))
    return data

Hàm này chạy một lượt, lưu CPU/RAM của API và CPU của PostgreSQL mỗi giây.
Sau tải, đối soát theo giao thức: số dư mỗi account khớp sổ cái, số dòng sổ cái mới khớp số request 201 (chuyển khoản 2 dòng mỗi request), không có số dư âm, tổng số dư không đổi sau `transfer`.
Kịch bản `mixed` cho ra hai dòng `mixed:login` và `mixed:list`, mỗi dòng là độ trễ riêng của nhóm đó.

In [ ]:
PG_PID = int(Path(PG_DATA, 'postmaster.pid').read_text().split()[0])

def pg_cpu_seconds():
    # Exited backends are reaped by the postmaster, so their time lands in its children_* counters.
    root = psutil.Process(PG_PID)
    t = root.cpu_times()
    total = t.user + t.system + t.children_user + t.children_system
    for child in root.children(recursive=True):
        try:
            c = child.cpu_times()
            total += c.user + c.system
        except psutil.NoSuchProcess:
            pass
    return total

def reconcile(workload, data, success):
    ids = [str(uuid.UUID(a)) for a in data['accountIds']]
    amount = int(settings['fixture']['transfer_amount' if workload == 'transfer' else 'debit_amount'])
    ledger = {}
    for line in sql("SELECT account_id, transaction_type, count(*) FILTER (WHERE reference_type <> 'OPENING_DEPOSIT'), "
                    "coalesce(sum(amount), 0) FROM financial_transactions GROUP BY 1, 2").splitlines():
        account_id, kind, count, total = line.split('|')
        ledger[account_id, kind] = (int(count), int(total.split('.')[0]))
    accounts = []
    for account_id, raw_id in zip(ids, data['accountIds']):
        final = int(api('GET', '/accounts/' + raw_id, data['token'])['currentBalance'])
        credit, debit = ledger.get((account_id, 'CREDIT'), (0, 0)), ledger.get((account_id, 'DEBIT'), (0, 0))
        accounts.append({'account_id': account_id, 'initial': INITIAL, 'final': final,
                         'new_credit_entries': credit[0], 'new_debit_entries': debit[0],
                         'ledger_ok': final == credit[1] - debit[1] and final >= 0})
    new_entries = sum(count for count, _ in ledger.values())
    total = sum(a['final'] for a in accounts)
    moved = {'debit': success, 'debit_spread': success}.get(workload, 0) * amount
    checks = {
        'ledger_matches_balance': all(a['ledger_ok'] for a in accounts),
        'no_negative_balance': all(a['final'] >= 0 for a in accounts),
        'new_entries_match_201': new_entries == success * (2 if workload == 'transfer' else 1 if workload.startswith('debit') else 0),
        'total_balance_ok': total == INITIAL * len(ids) - moved,
    }
    if workload == 'debit':
        checks['hot_account_only'] = all(a['new_debit_entries'] == 0 for a in accounts[1:])
    return {'checks': checks, 'new_entries': new_entries, 'total_balance': total, 'accounts': accounts}

def row_for(metrics, workload, tag=''):
    latency = metrics['operation_duration' + tag]['values']
    requests = metrics['operation_requests' + tag]['values']
    return {'workload': workload, 'requests': requests['count'], 'rps': requests['rate'],
            'p50_ms': latency['med'], 'p95_ms': latency['p(95)'], 'p99_ms': latency['p(99)'],
            'error_pct': metrics['operation_errors' + tag]['values']['rate'] * 100}

def run_test(workload, vus, duration, label):
    assert app.poll() is None, 'API has stopped'
    data = create_data()
    (OUT / (label + '-input.json')).write_text(json.dumps({
        'fixture': settings['fixture'], 'account_ids': data['accountIds'],
        'workload': workload, 'vus': vus, 'duration': duration}, indent=2))
    env = dict(os.environ, FIXTURE='/tmp/bank-data.json', PROTOCOL_FILE='/tmp/bank-protocol.json',
               WORKLOAD=workload, VUS=str(vus), DURATION=duration,
               SUMMARY=str(OUT / (label + '.json')))
    usage = []
    process = psutil.Process(app.pid)
    with open(OUT / (label + '.log'), 'w') as log:
        job = subprocess.Popen(['k6', 'run', '--quiet', '--no-usage-report', str(K6_SCRIPT)],
                               env=env, stdout=log, stderr=subprocess.STDOUT)
        try:
            pg_before, before = pg_cpu_seconds(), time.time()
            while job.poll() is None:
                cpu = process.cpu_percent(interval=1)
                pg_now, now = pg_cpu_seconds(), time.time()
                usage.append({'time': now, 'pid': app.pid, 'cpu_pct': cpu,
                              'pg_cpu_pct': (pg_now - pg_before) / (now - before) * 100,
                              'rss_mib': process.memory_info().rss / 1024**2})
                pg_before, before = pg_now, now
        finally:
            if job.poll() is None:
                job.terminate()
            job.wait(timeout=10)
    pd.DataFrame(usage).to_csv(OUT / (label + '-cpu.csv'), index=False)
    assert job.returncode in (0, 99), (OUT / (label + '.log')).read_text()[-3000:]
    assert len(usage) >= 2, 'Missing API resource samples'

    metrics = json.loads((OUT / (label + '.json')).read_text())['metrics']
    success = metrics['operation_successes']['values']['count']
    balance = reconcile(workload, data, success)
    (OUT / (label + '-balance.json')).write_text(json.dumps(balance, indent=2))
    common = {'vus': vus, 'duration': duration,
              'api_cpu_avg_pct': sum(s['cpu_pct'] for s in usage) / len(usage),
              'pg_cpu_avg_pct': sum(s['pg_cpu_pct'] for s in usage) / len(usage),
              'api_rss_max_mib': max(s['rss_mib'] for s in usage),
              'balance_ok': all(balance['checks'].values())}
    if workload == 'mixed':
        rows = [dict(row_for(metrics, 'mixed:' + name, '{name:' + name + '}'), **common) for name in ('login', 'list')]
    else:
        rows = [dict(row_for(metrics, workload), **common)]
    for row in rows:
        print(label, row, flush=True)
    assert common['balance_ok'], 'Reconciliation failed, discard this run; inspect ' + label + '-balance.json'
    return rows

## 9. Chạy tải
Smoke: mỗi kịch bản 5 giây ở mức tải thấp. Full: mỗi ô 60 giây, warm-up riêng 10 giây, lặp 3 lần, lượt chẵn chạy ngược thứ tự.
Muốn đo chính thức, chạy ô `MODE = 'full'` ở đây rồi chạy lại phần 9–11.

In [ ]:
assert MODE in ['smoke', 'full']
selected = WORKLOADS or list(settings['cases'])
assert set(selected) <= set(settings['cases']), selected
if MODE == 'smoke':
    smoke_vus = {'debit': 2, 'debit_spread': 2, 'transfer': 2, 'list': 1, 'login': 1, 'mixed': settings['cases']['mixed'][0]}
    cases = [(w, smoke_vus[w]) for w in selected]
    repeats, duration = 1, '5s'
else:
    cases = [(w, v) for w in selected for v in settings['cases'][w]]
    repeats, duration = settings['repeats'], settings['duration']

plan = {'mode': MODE, 'protocol': settings['id'], 'cases': cases, 'repeats': repeats, 'duration': duration,
        'warmup': settings['warmup'] if MODE == 'full' else None,
        'think_time_seconds': settings['think_time_seconds'], 'reverse_even_repeat': True}
(OUT / ('plan-' + MODE + '.json')).write_text(json.dumps(plan, indent=2))
rows = []
for repeat in range(1, repeats + 1):
    order = cases if repeat % 2 else list(reversed(cases))
    for workload, vus in order:
        label = f'{MODE}-{workload}-{vus}-run{repeat}'
        if MODE == 'full':
            run_test(workload, vus, settings['warmup'], label + '-warmup')
            time.sleep(settings['cooldown_seconds'])
        for row in run_test(workload, vus, duration, label):
            row['repeat'] = repeat
            rows.append(row)
        pd.DataFrame(rows).to_csv(OUT / ('results-' + MODE + '.csv'), index=False)
        time.sleep(settings['cooldown_seconds'])
display(pd.DataFrame(rows))

## 10. Thử rút vượt số dư
Cho 20 request cùng rút 10.000 VND từ account có 100.000 VND.
Mong đợi 10 lần thành công, 10 lần thiếu tiền, số dư 0 và 10 bút toán DEBIT.
Bài này chạy riêng, không cộng các response 409 vào error rate ở trên.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from threading import Barrier

data = create_data()
account = api('POST', '/accounts', data['token'],
              {'accountName': 'Race test', 'accountType': 'PAYMENT', 'initialDeposit': '100000'}, status=201)
barrier = Barrier(20)

def withdraw(_):
    barrier.wait(timeout=30)
    response = requests.post(BASE + '/accounts/' + account['id'] + '/transactions',
                             headers={'Authorization': 'Bearer ' + data['token']},
                             json={'type': 'DEBIT', 'amount': '10000'}, timeout=35)
    return response.status_code, response.json()

with ThreadPoolExecutor(max_workers=20) as pool:
    responses = list(pool.map(withdraw, range(20)))
success = sum(status == 201 and body.get('success') is True for status, body in responses)
rejected = sum(status == 409 and body.get('error', {}).get('code') == 'INSUFFICIENT_FUNDS' for status, body in responses)
balance = api('GET', '/accounts/' + account['id'], data['token'])['currentBalance']
account_id = str(uuid.UUID(account['id']))
entries = int(sql("SELECT count(*) FROM financial_transactions WHERE account_id='" + account_id + "' AND transaction_type='DEBIT'"))
race_result = {'success_201': success, 'insufficient_409': rejected, 'final_balance': balance, 'debit_entries': entries}
(OUT / ('race-' + MODE + '.json')).write_text(json.dumps(race_result, indent=2))
print(race_result)
assert (success, rejected, balance, entries) == (10, 10, '0', 10), 'Overdraft check failed'


## 11. Xem bảng kết quả và tải ZIP
Lấy trung vị từng chỉ số qua ba lượt, kèm min và max của p95 để thấy độ dao động; đây không phải percentile của dữ liệu gộp.
CPU 100% tương đương một CPU logic; RSS tính bằng MiB. Error threshold <1% chỉ là mốc thử nghiệm.
`summary-full.md` là bảng để dán vào `docs/performance/phase2-baseline.md`. ZIP giữ cả số liệu từng lượt, cấu hình máy, dữ liệu đầu vào, log và kiểm tra đối soát.

In [ ]:
df = pd.DataFrame(rows)
cols = ['rps', 'p50_ms', 'p95_ms', 'p99_ms', 'error_pct', 'api_cpu_avg_pct', 'pg_cpu_avg_pct', 'api_rss_max_mib']
grouped = df.groupby(['workload', 'vus'], sort=False)
summary = grouped[cols].median().join(grouped['p95_ms'].agg(p95_min='min', p95_max='max')).round(2)
display(summary)
summary.to_csv(OUT / ('summary-' + MODE + '.csv'))
head = ['workload', 'vus'] + list(summary.columns)
lines = ['| ' + ' | '.join(head) + ' |', '|' + ' --- |' * len(head)]
lines += ['| ' + ' | '.join([w, str(v)] + [f'{x:.2f}' for x in r]) + ' |' for (w, v), r in summary.iterrows()]
(OUT / ('summary-' + MODE + '.md')).write_text('\n'.join(lines) + '\n')  # pasted into docs/performance/phase2-baseline.md

report = {'mode': MODE, 'commit': COMMIT, 'protocol': settings['id'], 'expected_runs': len(cases) * repeats,
          'completed_runs': len(df[df['workload'] != 'mixed:list']),  # mixed yields two rows per run
          'balance_ok': all(r['balance_ok'] for r in rows),
          'error_threshold_pass': all(r['error_pct'] < 1 for r in rows),
          'overdraft_check': race_result}
(OUT / ('report-' + MODE + '.json')).write_text(json.dumps(report, indent=2))
archive = shutil.make_archive(str(OUT) + '-' + MODE, 'zip', OUT)
from IPython.display import FileLink
display(FileLink(archive))
print('Saved:', archive)

Nếu có lỗi khởi động, xem log bằng ô dưới. Không dùng kết quả smoke làm báo cáo hiệu năng.

In [ ]:
print((OUT / 'app.log').read_text()[-4000:])
